# 08 · Od přípravy dat k vyhodnocení modelu

Sestavte celý řetězec od kontroly dat po vyhodnocení modelu a předpovědi čtyř nových záznamů. Vlastní řešení pište do připravených prázdných kódových buněk pod jednotlivými úkoly.

Krátké spustitelné ukázky používají malá ilustrační data a proměnné s předponou `demo_`. Spouštějte je v pořadí: na sebe navazují a postupně ukazují celý pracovní postup. Doporučené nástroje a podrobnější nápovědy jsou u zadání.

Volitelný blok PCA vyřešte **před úkolem 7**; po otevření testu už konfiguraci neměňte. Po dokončení spusťte `Restart Kernel and Run All` a ověřte návaznost svého řešení.


## Příprava prostředí a načtení dat

1. Otevřete notebook s pracovním adresářem `08-Data-preprocessing`. Pokud používáte kořen repozitáře, přidejte tuto složku před relativní cesty k datům. Potřebné balíčky a spuštění popisuje README.
2. Spusťte připravené importy v ukázce níže; NumPy používá název `np` a pandas název `pd`. Pro všechna náhodná dělení nastavte `SEED` na **42**.
3. Načtěte `data/machine-records.csv` do tabulky `raw` a zobrazte několik prvních řádků. Původní tabulku ponechte nezměněnou.

**Co použít:** `pd.read_csv`, `head()` a `display` nebo `print`. Operace zapisujte do krátkých samostatných příkazů.

<details><summary>Nápověda: importy</summary>

Importovat lze zápisem `from modul import Název`. Nástroje patří do těchto modulů:

- `sklearn.model_selection`: `train_test_split`; `sklearn.base`: `clone`.
- `sklearn.dummy`: `DummyClassifier`; `sklearn.neighbors`: `KNeighborsClassifier`.
- `sklearn.impute`: `SimpleImputer`; `sklearn.preprocessing`: `StandardScaler`, `OneHotEncoder`.
- `sklearn.compose`: `ColumnTransformer`; `sklearn.pipeline`: `Pipeline`.
- `sklearn.metrics`: `balanced_accuracy_score`, `ConfusionMatrixDisplay`; pro volitelný blok `sklearn.decomposition`: `PCA`.

Pro zobrazení grafu lze importovat `matplotlib.pyplot` pod názvem `plt`. `display` je dostupné v Jupyteru, případně z `IPython.display`. Prázdná pole v CSV načte pandas jako chybějící hodnoty.

</details>

### Ukázka: Návodné ukázky: oddělená minidata


In [ ]:
from IPython.display import display  # Zobrazení tabulek ve vlastních úkolech.
import matplotlib.pyplot as plt  # Zobrazení grafu matice záměn.
from sklearn.metrics import ConfusionMatrixDisplay  # Matice skutečných a předpovězených tříd.
import numpy as np  # Číselné operace a značka chybějící hodnoty pro ukázky i vlastní práci.
import pandas as pd  # Tabulky ukázek jsou nezávislé na souborech cvičení.
from sklearn.model_selection import train_test_split  # Dělení ukázkových vstupů a cílů společně.
from sklearn.dummy import DummyClassifier  # Jednoduchý referenční klasifikátor.
from sklearn.base import clone  # Nová nenaučená kopie nastavení modelu.
from sklearn.impute import SimpleImputer  # Doplnění hodnot podle trénovacích dat.
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # Škálování čísel a kódování kategorií.
from sklearn.pipeline import Pipeline  # Spojení více kroků do jednoho řetězce.
from sklearn.compose import ColumnTransformer  # Různé transformace pro různé sloupce.
from sklearn.neighbors import KNeighborsClassifier  # Klasifikace podle sousedních trénovacích bodů.
from sklearn.metrics import balanced_accuracy_score  # Průměr úplností obou tříd.
from sklearn.decomposition import PCA  # Projekce číselných vstupů na hlavní komponenty.
demo_data = pd.DataFrame({  # Dvanáct umělých nezávislých vzorků; nejde o data zadání.
    "length": [1., 2., np.nan, 4., 5., 6., 7., 8., 9., 10., 11., 12.],  # První číselný příznak obsahuje jednu chybějící hodnotu.
    "mass": [12., 18., 35., 39., 52., 58., 71., 83., 87., np.nan, 108., 124.],  # Druhý číselný příznak má jiné měřítko a jednu chybějící hodnotu.
    "shape": ["round", "square"] * 6,  # Jedna neuspořádaná kategorie bez chybějících hodnot.
    "label": [0] * 6 + [1] * 6,  # Dvě stejně početné třídy pouze pro výuku syntaxe.
})  # Konec malé tabulky společné pro všechny ukázky.
print(demo_data.head(3))  # Krátký náhled vlastních dat ukázek.


**Vlastní kód:**


## Zadání a data

Jeden řádek představuje **samostatnou simulovanou situaci stroje**, nikoli sousední okamžik jedné časové řady. Situace jsou pro účely této úlohy nezávislé. Cíl `state` má dvě třídy: **0 = běžný simulovaný stav**, **1 = simulovaný stav ke kontrole**. Nejde o fyzikální model poruchy.

| Sloupce | Význam pro úlohu |
|---|---|
| `record_id` | identifikátor situace, pouze pro kontrolu duplicit |
| `temperature`, `vibration`, `pressure`, `power`, `speed`, `load` | šest číselných příznaků v odlišných měřítkách |
| `mode`, `environment` | provozní režim a prostředí; neuspořádané kategorie |
| `state` | cílová třída, nikdy vstup modelu |

CSV obsahuje 720 jedinečných situací a 6 duplicitních záznamů. Prázdná pole se načtou jako `np.nan`. Dokumentace sběru dat stanovuje dvě opravy: `temperature = -999` je značka chybějícího měření, `AUTO` znamená totéž jako `auto`. Tyto pevné opravy a odstranění identických duplicit předcházejí dělení; nic se při nich neodhaduje z dat. Duplicitní situace by se jinak mohla objevit v tréninku i testu.

Předpokládáme, že všech osm příznaků je dostupných **při předpovědi**. V reálné úloze by bylo nutné ověřit okamžik jejich dostupnosti. Statistické doplňování, škálování, kódování i PCA se odhadují až z tréninku. Testovací odlehlé hodnoty se podle chování modelu neodstraňují.

## 1. Prohlídka a známé opravy

**Cíl / výstup:** tabulka `clean` se 720 řádky bez duplicitního ID, bez `-999` v teplotě a bez kategorie `AUTO`. Souhrn `missing_counts` ukáže prázdné hodnoty **před** pevnými opravami; není to odhad parametrů předzpracování.

1. Doplnit počty prázdných hodnot v jednotlivých sloupcích a počet opakovaných `record_id`.
2. Odstranit opakované ID se zachováním prvního záznamu a vytvořit samostatnou kopii.
3. Změnit sentinel teploty na `np.nan` a sjednotit `AUTO` na `auto`.

**Co použít:** `isna()`, `sum()`, `duplicated(subset=...)`, `drop_duplicates(subset=..., keep="first")`, `copy()`, jednoduchý výběr `.loc[maska, "sloupec"]`. Každá operace může být na vlastním řádku; není potřeba `apply`, `lambda` ani `groupby`.

<details><summary>Nápověda</summary>

Výsledek `isna()` je tabulka hodnot ano/ne. Její `sum()` sečte hodnoty po sloupcích. `duplicated` vrací jednu hodnotu ano/ne pro každý řádek. Masku lze pojmenovat: `mask = table["column"] == value`; potom `table.loc[mask, "column"] = replacement`. Opravy nejsou odhadnuté z četností či kvantilů, jsou součástí dodané dokumentace.

</details>

**K zamyšlení:** Proč je duplicita stejné situace v tréninku i testu problém? Proč zatím nedoplňujeme prázdné hodnoty mediánem?

### Ukázka: prázdné hodnoty, duplicity a pevné opravy


In [ ]:
demo_dirty = pd.concat([demo_data, demo_data.iloc[[0]]], ignore_index=True)  # Přidání jedné přesné kopie řádku jen do pomocné tabulky.
demo_dirty.loc[1, "length"] = -1  # V ukázce značí záporná délka podle dokumentace chybějící hodnotu.
demo_missing = demo_dirty.isna().sum()  # Součet prázdných buněk zvlášť pro každý sloupec.
demo_duplicates = demo_dirty.duplicated().sum()  # Počet úplných kopií řádků za jejich prvním výskytem.
demo_clean = demo_dirty.drop_duplicates().copy()  # Odstranění kopií a samostatná tabulka pro opravy.
demo_bad_length = demo_clean["length"] < 0  # Maska vybere jen dokumentovanou neplatnou hodnotu.
demo_clean.loc[demo_bad_length, "length"] = np.nan  # Pevná oprava neodhaduje žádnou statistiku ze vzorků.
print(demo_missing, "duplicates:", demo_duplicates)  # Přehled původních nedostatků; další ukázky navazují na demo_clean.


**Vlastní kód:**


## 2. Příznaky a pevné dělení

**Cíl / výstup:** `X` obsahuje osm příznaků, `y` obsahuje `state`. Trénink / validace / test mají 432 / 144 / 144 řádků. `record_id` ani `state` do `X` nepatří.

1. Vytvořit `X` a `y`.
2. Odložit 20 % na test: vývojová část `X_dev`, `y_dev` a test `X_test`, `y_test`.
3. Z vývojových 80 % odložit čtvrtinu na validaci. Zbývá 60 % původních dat na trénink. Výsledky pojmenujte `X_train`, `X_val`, `y_train`, `y_val`.
4. V obou děleních použít `random_state=SEED` a stratifikaci podle příslušného cíle. Dělení se mezi pokusy nemění.
5. Ověřit, že se části nepřekrývají a že indexy vstupů odpovídají indexům jejich cílů. Zobrazit počty řádků a podíly tříd v tréninku a validaci.

**Co použít:** `drop(columns=[...])`, výběr cílového sloupce, `train_test_split(..., test_size=..., stratify=..., random_state=SEED)`. Funkce vrací vstupy pro první a druhou část, potom cíle pro první a druhou část.

<details><summary>Nápověda</summary>

Obecné pořadí je `A_train, A_hold, b_train, b_hold = train_test_split(A, b, ...)`. Při druhém dělení jsou vstupem pouze vývojová data; `stratify` proto dostává vývojový cíl. Čtvrtina z 80 % představuje 20 % původního souboru. Stratifikace přibližně zachovává poměry tříd, nevytváří stejně početné třídy.

</details>

**K zamyšlení:** Proč má druhé dělení `test_size=0.25`, přestože validace tvoří jen 20 % všech dat?

### Ukázka: společné dělení vstupů a cíle


In [ ]:
demo_X = demo_clean.drop(columns="label")  # Vstup obsahuje dvě čísla a jednu kategorii, nikoli cílovou třídu.
demo_y = demo_clean["label"]  # Cíl je oddělený od vstupních příznaků.
demo_X_dev, demo_X_test, demo_y_dev, demo_y_test = train_test_split(demo_X, demo_y, test_size=0.25, stratify=demo_y, random_state=7)  # Tři vzorky se odloží na test ještě před učením.
demo_X_train, demo_X_val, demo_y_train, demo_y_val = train_test_split(demo_X_dev, demo_y_dev, test_size=1/3, stratify=demo_y_dev, random_state=7)  # Z devíti vývojových vzorků jsou tři validační; šest zbývá na trénink.
print(len(demo_X_train), len(demo_X_val), len(demo_X_test))  # Malé počty ukazují syntaxi, neposkytují spolehlivý odhad kvality.


**Vlastní kód:**


## 3. Referenční klasifikátor

**Cíl / výstup:** `baseline` předpovídá nejčastější trénovací třídu. `baseline_score` je jeho validační **vyvážená správnost (balanced accuracy)**: průměr úplností obou tříd. Obě třídy tak mají v metrice stejnou váhu, i když mají různou četnost.

Vytvořit `DummyClassifier(strategy="most_frequent")`, naučit ho pouze na tréninku, předpovědět validaci a spočítat metriku. Tento klasifikátor ignoruje příznaky, proto nepotřebuje doplňování ani kódování.

**Co použít:** konstruktor, `.fit(vstupy, cíl)`, `.predict(vstupy)` a `balanced_accuracy_score(skutečnost, předpověď)`.

<details><summary>Nápověda</summary>

Učení a předpověď jsou dva samostatné příkazy. `fit` dostává vstupy i správné odpovědi; `predict` pouze vstupy. Validační `y` slouží až k výpočtu metriky, ne k učení. Konstantní předpověď při zastoupení obou tříd dosáhne balanced accuracy 0,5.

</details>

**K zamyšlení:** Proč může mít konstantní předpověď vysokou běžnou správnost, ale balanced accuracy jen 0,5?

### Ukázka: referenční předpověď a metrika


In [ ]:
demo_baseline = DummyClassifier(strategy="most_frequent")  # Nastavení konstantní předpovědi nejčastější trénovací třídy.
demo_baseline.fit(demo_X_train, demo_y_train)  # Tento model používá jen četnosti tříd a příznaky ignoruje.
demo_baseline_pred = demo_baseline.predict(demo_X_val)  # Předpověď dostává vstupy bez validačních odpovědí.
print(balanced_accuracy_score(demo_y_val, demo_baseline_pred))  # Skutečné validační cíle vstupují až do hodnocení.


**Vlastní kód:**


## 4. Dvě větve předzpracování a model

**Cíl / výstup:** nenaučený `pipeline`, který přijme původních osm sloupců a provede celé předzpracování i klasifikaci. V tomto kroku se ještě nic neučí.

Vytvořte seznam `NUMERIC` se sloupci `temperature`, `vibration`, `pressure`, `power`, `speed`, `load` a seznam `CATEGORICAL` se sloupci `mode`, `environment`.

| Objekt | Povinné kroky a jejich názvy |
|---|---|
| `numeric_pipe` | `impute`: `SimpleImputer(strategy="median")` → `scale`: `StandardScaler()` |
| `categorical_pipe` | `impute`: `SimpleImputer(strategy="most_frequent")` → `encode`: `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` |
| `prepare` | `ColumnTransformer` s větví `num` pro `NUMERIC` a `cat` pro `CATEGORICAL` |
| `pipeline` | `prepare`: předchozí transformátor → `model`: `KNeighborsClassifier(n_neighbors=7)` |

Číselné příznaky mají různé měřítko; kNN porovnává vzdálenosti. Kategorie `mode` a `environment` nemají přirozené pořadí. One-hot proto každé známé kategorii vytvoří vlastní sloupec. `sparse_output=False` zde vrací obyčejnou hustou matici pro malý dataset. Neznámá kategorie při predikci díky `handle_unknown="ignore"` nevyvolá chybu; v příslušném bloku bude mít samé nuly.

**Co použít:** `Pipeline`, `ColumnTransformer`, konstruktory v tabulce a vytvořené seznamy sloupců. Názvy kroků je potřeba dodržet; použijí se v dalších úkolech.

<details><summary>Nápověda</summary>

Syntaxe řetězce je `Pipeline([("jméno_kroku", objekt), ...])`. Větev ve `ColumnTransformer` má tři části: `("jméno_větve", transformátor, seznam_sloupců)`. Konstruktor pouze popíše postup. `fit` na celém řetězci později odhadne statistiky a kategorie na tréninku a naučí model. Není nutné ručně převádět mezivýsledky zpět do pandas.

</details>

Dokumentace: [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

**K zamyšlení:** Proč dostávají číselné a kategoriální sloupce jiné kroky? Proč zde nepoužíváme kódy `auto=0`, `manual=1`, `eco=2` jako číselné vzdálenosti?

### Ukázka: číselná a kategoriální větev


In [ ]:
demo_numeric = Pipeline([  # Návazné kroky pro dva číselné sloupce.
    ("fill", SimpleImputer(strategy="mean")),  # Průměr se naučí až při fit a pouze z tréninku.
    ("scale", StandardScaler()),  # Škálování následuje po doplnění chybějících hodnot.
])  # Konec číselné větve.
demo_prepare = ColumnTransformer([  # Každá trojice uvádí název, transformaci a sloupce.
    ("numbers", demo_numeric, ["length", "mass"]),  # Dva číselné sloupce používají stejnou větev.
    ("shapes", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["shape"]),  # Kategorie jsou zde úplné, proto ukázka před kódováním nepotřebuje imputaci.
])  # Konec transformátoru sloupců.
demo_pipeline = Pipeline([  # Celý řetězec bude přijímat původní tři vstupní sloupce.
    ("inputs", demo_prepare),  # Předzpracování je součást stejného učení jako model.
    ("classifier", KNeighborsClassifier(n_neighbors=1)),  # Malá ukázka používá jediného nejbližšího souseda.
])  # Konstrukce dosud neodhadla žádné parametry z dat.


**Vlastní kód:**


## 5. Učení celého řetězce a validace

**Cíl / výstup:** naučený `fitted_pipeline`, jeho validační předpovědi `val_pred` a `val_score`.

Vytvořit nenaučenou kopii konfigurace pomocí `clone`, naučit **celý** řetězec jen na `X_train`, `y_train` a předpovědět původní tabulku `X_val`. Validace se samostatně neučí ani znovu neškáluje. Výsledek porovnat s baseline stejnou metrikou. Zaznamenat názvy `baseline` a `standard` a jejich validační skóre do malé tabulky; lze použít `pd.DataFrame` ze slovníku seznamů.

**Co použít:** `clone`, `.fit`, `.predict`, `balanced_accuracy_score`.

<details><summary>Nápověda</summary>

`copy = clone(configuration)` zachová nastavení, nikoli naučené statistiky. `fit` se volá na této kopii celého řetězce. Její `predict` automaticky použije stejné doplnění, škálování a kategorie jako při učení. Předpověď proto dostává osm původních sloupců, nikoli ručně transformovanou matici.

</details>

Souvislost: [únik informace a konzistentní předzpracování](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

**K zamyšlení:** Jak dopadl model vůči baseline? Které části řetězce by unikaly informace, kdyby se jejich `fit` provedl před dělením?

### Ukázka: učení a předpověď původní tabulky


In [ ]:
demo_fitted = clone(demo_pipeline)  # Nenaučená kopie zachová nastavení všech kroků.
demo_fitted.fit(demo_X_train, demo_y_train)  # Doplnění, škálování, kategorie i model se naučí pouze z tréninku.
demo_val_pred = demo_fitted.predict(demo_X_val)  # Validace projde již naučenými transformacemi bez nového fit.
demo_val_score = balanced_accuracy_score(demo_y_val, demo_val_pred)  # Stejná metrika jako u referenčního modelu.
print(demo_val_score)  # Ukázkové skóre je založené jen na třech validačních vzorcích.


**Vlastní kód:**


## 6. Jeden kontrolovaný experiment

**Cíl / výstup:** `experiment` používá stejnou imputaci, kódování, model i dělení, ale **bez škálování číselných příznaků**. `experiment_score` umožní porovnání s `val_score`.

Vytvořit další `clone(pipeline)`. Jedinou změnou je nahrazení kroku škálování hodnotou `"passthrough"`. Pak následuje učení na stejném tréninku, predikce stejné validace a stejná metrika. Původní naučený řetězec se nepřepisuje. Do srovnání přidat variantu `unscaled` a její validační skóre.

**Co použít:** `clone`, `.set_params(prepare__num__scale="passthrough")`, `.fit`, `.predict`, `balanced_accuracy_score`. Dvojité podtržítko odděluje názvy vnořených kroků.

<details><summary>Nápověda</summary>

`set_params` mění konfiguraci před učením. `passthrough` znamená „tento krok vynechat“, nikoli vyřadit číselné sloupce. Postup je kopie → jedna změna → nové učení → validace. Není zaručeno, že každá rozumná úprava zlepší skóre; i stejný nebo horší výsledek je platný experiment.

</details>

Volitelně lze provést **jiný samostatný pokus**: v nové kopii původního řetězce změnit pouze `prepare__num__impute__strategy="mean"`. Nelze současně změnit imputaci i škálování a připsat rozdíl jediné metodě.

**K zamyšlení:** Před pokusem: který výsledek očekáváte a proč? Po pokusu: co ukazuje rozdíl validačních skóre a co z něj nelze zobecnit?

### Ukázka: kopie a změna jediného parametru


In [ ]:
demo_experiment = clone(demo_pipeline)  # Samostatná konfigurace ponechá první model beze změny.
demo_experiment.set_params(classifier__n_neighbors=3)  # Dvojité podtržítko vybírá parametr kroku classifier; mění se jen počet sousedů.
demo_experiment.fit(demo_X_train, demo_y_train)  # Nové učení používá stejné trénovací řádky.
demo_experiment_pred = demo_experiment.predict(demo_X_val)  # Stejné validační řádky umožňují srovnání.
demo_experiment_score = balanced_accuracy_score(demo_y_val, demo_experiment_pred)  # Cíle a metrika se mezi pokusy nemění.
print(demo_val_score, demo_experiment_score)  # Rozdíl přísluší této jediné změně v tomto malém dělení.


**Vlastní kód:**


## Volitelně: PCA v číselné větvi

**Cíl / výstup:** `pca_model` a `pca_score`. Pokud tento blok řešíte, přidejte variantu `pca` do validačního srovnání před závěrečným testem. Zobrazte také součet `explained_variance_ratio_` naučené PCA.

Z původního řetězce vytvořit kopii. Její číselnou větev nahradit řetězcem **impute (median) → scale (StandardScaler) → pca (PCA se 3 komponentami)**. Kategoriální větev a kNN se sedmi sousedy zůstávají stejné. Celý postup znovu naučit pouze na tréninku a vyhodnotit na validaci.

**Co použít:** `Pipeline`, `PCA(n_components=3)`, `clone`, `.set_params(prepare__num=...)`, `.fit`, `.predict`, `balanced_accuracy_score`.

<details><summary>Nápověda</summary>

PCA přijímá doplněná a přeškálovaná čísla. Patří proto za tyto dva kroky uvnitř `num`, nikoli za výsledné one-hot sloupce. Sestavení je stejné jako v úkolu 4, jen přibude krok pojmenovaný `pca`. Tři komponenty nahrazují šest číselných příznaků; kategoriální sloupce zůstávají. `explained_variance_ratio_` popisuje zachovaný rozptyl, nikoli úspěšnost klasifikace.

Naučenou číselnou větev získáte z `pca_model.named_steps["prepare"].named_transformers_["num"]`; uložte ji do vlastní proměnné. Z ní vyberte `named_steps["pca"]` a sečtěte jeho `explained_variance_ratio_`. Původní objekt před učením nestačí: transformátor při učení vytváří naučené kopie větví.

</details>

Tento pokus nevyužívá test. Pokud už proběhl úkol 7, bonus se nyní nepoužije k dodatečné změně zvoleného modelu.

**K zamyšlení:** Jak se změnila validační kvalita a počet číselných vstupů modelu? Proč samotný zachovaný rozptyl nestačí pro volbu klasifikátoru?

### Ukázka: jedna komponenta ze dvou čísel


In [ ]:
demo_pca_numeric = Pipeline([  # Nová číselná větev navazuje na stejnou přípravu.
    ("fill", SimpleImputer(strategy="mean")),  # Stejné doplnění chybějících čísel jako v původní variantě.
    ("scale", StandardScaler()),  # Stejné škálování před projekcí.
    ("reduce", PCA(n_components=1)),  # Dvě číselné veličiny se promítnou do jedné hlavní komponenty.
])  # Kategoriální indikátory zůstanou mimo PCA.
demo_pca_model = clone(demo_pipeline)  # Samostatná kopie původní konfigurace.
demo_pca_model.set_params(inputs__numbers=demo_pca_numeric)  # Náhrada pouze číselné větve, ostatní kroky jsou stejné.
demo_pca_model.fit(demo_X_train, demo_y_train)  # Také směr PCA se odhaduje jen z tréninku.
demo_pca_pred = demo_pca_model.predict(demo_X_val)  # Stejná validace jako u předchozích variant.
demo_pca_score = balanced_accuracy_score(demo_y_val, demo_pca_pred)  # Kvalitu klasifikace určuje metrika, nikoli samotný zachovaný rozptyl.
print(demo_pca_score)  # Výsledek se porovnává ještě před použitím testu.


**Vlastní kód:**


## 7. Uzavření výběru a konečný test

**Cíl / výstup:** zvolený název `chosen_name`, nově naučený `final_pipeline`, jediné testovací předpovědi `test_pred` a jejich `test_score` s maticí záměn.

Použijte vlastní tabulku validačních skóre z předchozích úkolů. Z dvojice `standard` a `unscaled`, případně i předem dokončeného `pca`, zvolit **nejvyšší nezaokrouhlenou validační balanced accuracy**. Při shodě se přednostně ponechá `standard`, poté `unscaled`, poté `pca`. Jde o jednoduché pravidlo tohoto cvičení, nikoli důkaz obecně nejlepší metody.

Zvolený název uložte do `chosen_name` a odpovídající objekt do `selected_pipeline`: `standard` odpovídá `fitted_pipeline`, `unscaled` objektu `experiment` a volitelné `pca` objektu `pca_model`.

**Co použít:** připravené objekty z vlastních předchozích výpočtů, `pd.concat([část1, část2])`, `clone`, `.fit`, `.predict`, `balanced_accuracy_score` a `ConfusionMatrixDisplay.from_predictions` pro matici záměn.

<details><summary>Nápověda</summary>

`chosen_name` je jeden z názvů v tabulce, například textový řetězec. Vybraný objekt se znovu nepoužije s původními statistikami: `clone` vytvoří čistou konfiguraci. Trénink a validace se spojí zvlášť pro `X` a zvlášť pro `y` ve stejném pořadí. Celý nový řetězec se naučí na těchto 576 situacích. Teprve potom přijde jediná predikce `X_test`; jejich cíl se použije jen k vyhodnocení.

</details>

**Po zobrazení testu už následuje pouze interpretace a předpověď nových dat.** Volba varianty, prahů nebo odstranění řádků podle testových výsledků by změnila test na další validaci. Opětovné spuštění pevného dokončeného řešení kvůli reprodukovatelnosti není nový experiment.

**K zamyšlení:** Kolik situací třídy 1 model označil jako 0? Jak se liší testové a validační skóre a proč tento rozdíl neopravujeme dalším laděním na testu?

### Ukázka: výběr podle validace a nové učení


In [ ]:
demo_candidates = {"one": demo_fitted, "three": demo_experiment}  # Dvě hlavní konfigurace; volitelná ukázka PCA není potřebná.
demo_scores = {"one": demo_val_score, "three": demo_experiment_score}  # Výběr používá nezaokrouhlené validační hodnoty.
demo_choice = max(demo_scores, key=demo_scores.get)  # Nejvyšší skóre; při shodě první vložený název.
demo_final = clone(demo_candidates[demo_choice])  # Zvolená konfigurace začne bez starých naučených statistik.
demo_final.fit(demo_X_dev, demo_y_dev)  # Nové učení celého řetězce na devíti vývojových vzorcích.
demo_test_pred = demo_final.predict(demo_X_test)  # Test vstupuje do předpovědi až po uzavření výběru.
demo_test_score = balanced_accuracy_score(demo_y_test, demo_test_pred)  # Po dělení se testové cíle používají pouze k tomuto závěrečnému hodnocení.
print(demo_choice, demo_test_score)  # Tři testové vzorky slouží jen k demonstraci; podle výsledku se nic neladí.


**Vlastní kód:**


## 8. Čtyři nové záznamy

**Cíl / výstup:** čtyři předpovědi `new_pred` z již naučeného `final_pipeline`. Nová data nemají sloupec `state`, a proto u nich nelze spočítat správnost.

Načtěte `data/new-records.csv` do tabulky `new_data`. Má stejné vstupní schéma a čtyři nové situace. Jedna používá dosud neznámý režim `service`. Z tabulky vybrat stejné vstupní sloupce jako v `X` a předpovědět je celým finálním řetězcem. **Žádný další `fit` ani nové sestavení kategorií.** Zobrazte tabulku se sloupcem `record_id` a odpovídajícími předpověďmi; ID slouží pouze k jejich označení.

**Co použít:** výběr `new_data[X.columns]` a `.predict` na finálním řetězci.

<details><summary>Nápověda</summary>

Vstupem modelu zůstává původní tabulka, ne ručně kódované sloupce. Neznámý režim se díky nastavení enkodéru převede na nuly v bloku `mode`; neznamená to, že model poznal význam `service` nebo že jeho předpověď musí být spolehlivá. Předpovědi jsou simulovaný výstup pro následné posouzení.

</details>

**K zamyšlení:** Proč neznámá kategorie nevyvolala chybu? Co by ještě bylo potřeba pro posouzení správnosti těchto čtyř předpovědí?

### Ukázka: nové vzorky bez dalšího učení


In [ ]:
demo_new = pd.DataFrame({"length": [2.5, 10.5], "mass": [26., np.nan], "shape": ["round", "oval"]})  # Dva nové vzorky se stejnými vstupy; oval je neznámá kategorie.
demo_new_pred = demo_final.predict(demo_new)  # Již naučený řetězec doplní hodnoty i zakóduje kategorie bez nového fit.
print(demo_new_pred)  # Bez skutečných cílů lze ukázat předpovědi, nikoli jejich správnost.


**Vlastní kód:**
